# HU-2.2 — Fine-tune YOLO11 on football-players-detection

**Runtime → Change runtime type → GPU (T4)**

Dataset: [football-players-detection v1](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc/dataset/1) (CC BY 4.0)\n
Classes: `ball`, `goalkeeper`, `player`, `referee`

After training, download `best.pt` into your local repo as `models/best_yolov11n.pt`.


## 0. Check GPU


In [ ]:
!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## 1. Install deps


In [ ]:
!pip -q install -U ultralytics roboflow
import ultralytics
ultralytics.checks()


## 2. Download dataset

In Colab: 🔑 icon (Secrets) → add `ROBOFLOW_API_KEY` → enable notebook access.\n
Or paste the key in the cell below (less safe).


In [ ]:
import os
from pathlib import Path

# Prefer Colab Secrets
try:
    from google.colab import userdata
    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    print("Loaded ROBOFLOW_API_KEY from Colab Secrets")
except Exception as e:
    print("Secrets not available:", e)
    # FALLBACK (optional): uncomment and paste once, then delete the cell output
    # os.environ["ROBOFLOW_API_KEY"] = "rf_..."

assert os.environ.get("ROBOFLOW_API_KEY"), "Set ROBOFLOW_API_KEY in Colab Secrets"

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(1)
dataset = version.download("yolov8")
print("dataset:", dataset.location)
print(Path(dataset.location, "data.yaml").read_text())


## 3. Normalize data.yaml paths (safer for Ultralytics)


In [ ]:
from pathlib import Path
import yaml

data_yaml = Path(dataset.location) / "data.yaml"
cfg = yaml.safe_load(data_yaml.read_text())
# Force paths relative to the yaml directory
cfg["path"] = str(Path(dataset.location).resolve())
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
cfg["test"] = "test/images"
# Keep names as list or dict — Ultralytics accepts list
if isinstance(cfg.get("names"), dict):
    pass
data_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(data_yaml.read_text())


## 4. Train YOLO11n

Start small (`epochs=50`) to validate the setup; raise to 100 later if metrics look good.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(
    data=str(Path(dataset.location) / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="runs/football",
    name="yolo11n_ft",
    seed=42,
    patience=20,
    exist_ok=True,
)
print("done")


## 5. Validate on test split


In [ ]:
best = Path("runs/football/yolo11n_ft/weights/best.pt")
assert best.exists(), best
metrics = YOLO(str(best)).val(
    data=str(Path(dataset.location) / "data.yaml"),
    split="test",
    imgsz=640,
    device=0,
)
print(metrics)


## 6. Download weights to your machine


In [ ]:
from google.colab import files
import shutil

out = Path("best_yolov11n.pt")
shutil.copy(best, out)
print("Downloading", out, "size_mb", round(out.stat().st_size/1e6, 2))
files.download(str(out))

# Also zip useful plots
shutil.make_archive("yolo11n_ft_run", "zip", "runs/football/yolo11n_ft")
files.download("yolo11n_ft_run.zip")


## 7. Local next steps

```bash
# after download:
mv ~/Downloads/best_yolov11n.pt ~/Desktop/football-cv-pipeline/models/
# then predict on sample:
cd ~/Desktop/football-cv-pipeline
source .venv/bin/activate
yolo detect predict model=models/best_yolov11n.pt source=input_videos/sample.mp4 imgsz=640 conf=0.2 device=mps project=runs/ft name=yolo11n_sample
```

